# Revision add-ons (R1–R5) — chạy trên kết quả FROZEN có sẵn, không đổi mô hình/T10
Chạy từng ô bằng Shift+Enter. Kết quả ghi vào `10_revision_out/`.

In [ ]:
from pathlib import Path
import os, sys, importlib.util
PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
os.chdir(PROJECT_ROOT)
spec = importlib.util.spec_from_file_location("rv", PROJECT_ROOT / "10_revision" / "revision_addons.py")
rv = importlib.util.module_from_spec(spec); spec.loader.exec_module(rv)
print("OK", PROJECT_ROOT)

## 1. Tìm file kết quả frozen (dev predictions, T10 predictions, panel)

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 250)
cand = rv.discover(".")
display(cand.sort_values("MB", ascending=False).head(60))

## 2. Điền CONFIG theo kết quả ô trên
- `dev_pred_path`: file dự đoán dev T4–T9 của **tất cả** feature set (A0–A10), hoặc dùng `dev_pred_files` = {"A1": path, ...}
- `t10_pred_path`: dự đoán T10 của mô hình frozen
- `panel_path`: bảng tích hợp (listing × origin) có TA code, nhãn, 4 biến construction, 2 biến rental
- tên cột: `id_col`, `t_col`, `y_col`, `p_col`, `ta_col`

In [ ]:
import pandas as pd
from pathlib import Path
Path("10_revision_out").mkdir(exist_ok=True)

# 1) Panel nhỏ: chỉ các cột cần, thêm nhãn y_disappear = 1 - persistence
cols = ["transition", "listing_id", "ta_code", "next_period_persistence",
        "construction_dwelling_3m_log", "construction_dwelling_12m_log",
        "construction_momentum_3m_vs_prior9m", "construction_multiunit_share_12m",
        "rental_geomean_rent_log", "rental_geomean_rent_change_12m_log"]
pan = pd.read_parquet("02_intermediate/modelling/modelling_panel_T1_T10.parquet", columns=cols)
pan["y_disappear"] = 1 - pan["next_period_persistence"].astype(int)
pan = pan.drop(columns="next_period_persistence")
pan.to_parquet("10_revision_out/panel_small.parquet", index=False)
print("panel:", pan.shape, "| disappear total:", int(pan.y_disappear.sum()))

# 2) T10 predictions: đổi tên cột xác suất raw cho khớp với dev
t10 = pd.read_parquet("05_results/final_locked_T10_primary/predictions/J_final_locked_T10_primary_predictions.parquet",
                      columns=["transition", "listing_id", "y_disappear", "p_disappear_raw"])
t10 = t10.rename(columns={"p_disappear_raw": "p_disappear"})
t10.to_parquet("10_revision_out/t10_small.parquet", index=False)
print("T10:", t10.shape, "| events:", int(t10.y_disappear.sum()))

In [ ]:
rv.CONFIG.update(
    dev_pred_path = "05_results/development_modelling_IB/predictions/IB_all_development_predictions.parquet",
    dev_spec_col  = "feature_set",
    dev_model_col = "model", dev_model_value = "xgb",
    dev_pred_files = {},
    spec_name_map = {},
    t10_pred_path = "10_revision_out/t10_small.parquet",
    panel_path    = "10_revision_out/panel_small.parquet",
    id_col="listing_id", t_col="transition", y_col="y_disappear", p_col="p_disappear", ta_col="ta_code",
    construction_cols = ["construction_dwelling_3m_log", "construction_dwelling_12m_log",
                         "construction_momentum_3m_vs_prior9m", "construction_multiunit_share_12m"],
    construction_level_col = "construction_dwelling_12m_log",
    rental_cols = ["rental_geomean_rent_log", "rental_geomean_rent_change_12m_log"],
    bootstrap_reps = 400,
)
dev = rv.load_dev(); pan = rv.load_panel()
display(dev.groupby(["spec", "t"]).size().unstack())

## 3. Chạy từng phân tích (R1 lâu nhất, ~15–40 phút)

In [ ]:
r1 = rv.family_paired(dev, pan); display(r1)

In [ ]:
r2 = rv.confirmed_label(dev, 'A5'); display(r2)

In [ ]:
r3 = rv.construction_strata(dev, pan); display(r3)

In [ ]:
r4 = rv.ta_association(pan); display(r4)

In [ ]:
rv.write_latex(r1, r2, r3, r4); rv.shareable_package(dev, pan)
print(open('10_revision_out/revision_macros.tex').read())

## 4. Đưa vào manuscript
Chép `10_revision_out/revision_macros.tex`, `tab_family_paired.tex`, `tab_confirmed_label.tex`, `tab_construction_strata.tex`
vào cùng thư mục với `main.tex` (bản sửa trong `10_revision/manuscript/`), rồi compile:
`pdflatex main; bibtex main; pdflatex main; pdflatex main`.
**Đọc lại 4 đoạn có số mới** (§4.5, §4.6, §4.7, Abstract) — nếu khoảng tin cậy chứa 0, giữ nguyên câu chữ trung tính; nếu không, có thể nói mạnh hơn.